# Extract payout addresses (the expensive, one-time pull)

To add the second attribution method (H2) and the heuristic axis (H1-H5), we need
each block's coinbase **payout addresses**, which live in the big `transactions`
table.

**Cost-first design.** A single recent month can scan tens of GB (the ordinals
era), so we never run blindly:

1. **Phase 1** dry-runs *every year* (free, no scan) and shows the total
2. **Phase 2** extracts year-by-year and caches each year. 

Then we merge the addresses with cached tag data and attribute every block
with both methods.

In [1]:
%matplotlib inline
import os, glob
import numpy as np
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "project-9536b219-93fb-4fca-aac"  
client = bigquery.Client(project=PROJECT_ID)

YEARS = range(2009, 2027)   

def address_query(year):
    return f"""
    SELECT
      block_number AS height,
      ARRAY(SELECT addr FROM UNNEST(outputs) AS o, UNNEST(o.addresses) AS addr) AS output_addresses
    FROM `bigquery-public-data.crypto_bitcoin.transactions`
    WHERE is_coinbase = TRUE
      AND block_timestamp_month >= '{year}-01-01'
      AND block_timestamp_month <  '{year + 1}-01-01'
    """

print("client ready:", client.project)

client ready: project-9536b219-93fb-4fca-aac


## Phase 1 - dry-run every year (free), see the total



In [2]:
BUDGET_GB = 300   

est = {}
for y in YEARS:
    dry = client.query(address_query(y), bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    est[y] = dry.total_bytes_processed / 1e9

for y in YEARS:
    print(f"  {y}: {est[y]:8.2f} GB   " + "#" * int(est[y] / 5))
total = sum(est.values())
print("-" * 30)
print(f"  TOTAL one-time scan: {total:.0f} GB  ({total/1000:.2f} TB).  Free tier = 1000 GB/month.")
assert total < BUDGET_GB, f"Total {total:.0f} GB exceeds budget -- split the heavy years into months."

  2009:     0.00 GB   
  2010:     0.01 GB   
  2011:     0.20 GB   
  2012:     0.85 GB   
  2013:     2.11 GB   
  2014:     3.26 GB   
  2015:     6.13 GB   #
  2016:     9.08 GB   #
  2017:    11.43 GB   ##
  2018:     9.46 GB   #
  2019:    13.64 GB   ##
  2020:    13.31 GB   ##
  2021:    13.71 GB   ##
  2022:    14.01 GB   ##
  2023:    22.51 GB   ####
  2024:    28.70 GB   #####
  2025:    22.60 GB   ####
  2026:    16.20 GB   ###
------------------------------
  TOTAL one-time scan: 187 GB  (0.19 TB).  Free tier = 1000 GB/month.


## Phase 2 - extract and cache, year by year


In [3]:
os.makedirs("../data/raw/addresses", exist_ok=True)
scanned = 0.0

for y in YEARS:
    path = f"../data/raw/addresses/addresses_{y}.parquet"
    if os.path.exists(path):
        print(f"  {y}: cached, skipping")
        continue
    job = client.query(address_query(y))
    df = job.to_dataframe()
    gb = job.total_bytes_processed / 1e9
    scanned += gb
    df.to_parquet(path)
    print(f"  {y}: {len(df):>7,} coinbase blocks  |  scanned {gb:6.1f} GB  (running {scanned:6.1f} GB)")
print("done")

  2009: cached, skipping
  2010: cached, skipping
  2011: cached, skipping
  2012: cached, skipping
  2013: cached, skipping
  2014: cached, skipping
  2015: cached, skipping
  2016: cached, skipping
  2017: cached, skipping
  2018: cached, skipping
  2019: cached, skipping
  2020: cached, skipping
  2021: cached, skipping
  2022: cached, skipping
  2023: cached, skipping
  2024: cached, skipping
  2025: cached, skipping
  2026: cached, skipping
done


## Combine and merge with the tag data


In [4]:
parts = [pd.read_parquet(p) for p in sorted(glob.glob("../data/raw/addresses/addresses_*.parquet"))]
addresses = pd.concat(parts, ignore_index=True)

raw = pd.read_parquet("../data/raw/raw_blocks.parquet")
full = raw.merge(addresses, on="height", how="left")
full.to_parquet("../data/raw/raw_blocks_full.parquet")

has_addr = full["output_addresses"].apply(lambda a: isinstance(a, (list, np.ndarray)))
print(f"{len(full):,} blocks total | matched to an address record: {has_addr.mean():.1%}")

954,375 blocks total | matched to an address record: 100.0%


## Attribute every block with BOTH methods

Running tag matching and address matching together, and combine them with the NA-safe
confidence scheme all from tested modules. This is the full two-method
attribution over the whole chain.

In [5]:
try:
    from stage2_attribute.reference_loader import load_reference
    from stage2_attribute.tag_matcher import TagMatcher
    from stage2_attribute.address_matcher import AddressMatcher
    from stage2_attribute.confidence import confidence_frame
except ModuleNotFoundError:
    import sys
    sys.path.insert(0, os.path.abspath("../src"))
    from stage2_attribute.reference_loader import load_reference
    from stage2_attribute.tag_matcher import TagMatcher
    from stage2_attribute.address_matcher import AddressMatcher
    from stage2_attribute.confidence import confidence_frame

coinbase_tags, payout_addresses = load_reference("../data/raw/pools.json")
tm = TagMatcher(coinbase_tags)
am = AddressMatcher(payout_addresses)

full["tag_pool"] = full["coinbase_param"].apply(tm.match_hex)
full["addr_pool"] = full["output_addresses"].apply(
    lambda a: am.match(a) if isinstance(a, (list, np.ndarray)) else None)
pool, conf = confidence_frame(full["tag_pool"], full["addr_pool"])
full["pool"], full["confidence"] = pool, conf

full[["height", "timestamp", "tag_pool", "addr_pool", "pool", "confidence"]] \
    .to_parquet("../data/derived/attributed_full.parquet")

print(full["confidence"].value_counts().to_string())
tag_cov = full["tag_pool"].notna().mean()
addr_cov = full["addr_pool"].notna().mean()
either = (full["tag_pool"].notna() | full["addr_pool"].notna()).mean()
print(f"\ntag coverage:     {tag_cov:.1%}")
print(f"address coverage: {addr_cov:.1%}")
print(f"either method:    {either:.1%}   (tag-only was {tag_cov:.1%} -> address adds "
      f"{either - tag_cov:.1%})")

confidence
TAG_ONLY     388341
HIGH         278922
UNKNOWN      226683
ADDR_ONLY     59283
CONFLICT       1146

tag coverage:     70.0%
address coverage: 35.6%
either method:    76.2%   (tag-only was 70.0% -> address adds 6.2%)
